# Data Download

This notebook downloads the requisite data needed to draft fantasy baseball players. It creates a data directory and stores the data that is easily and quickly digestable into PANDAS DataFrames

## Import libraries, configure seasons, and enable PyBaseball cache


In [1]:
from __future__ import annotations

from datetime import date
from typing import Dict, List

import pandas as pd
from pybaseball import (
    batting_stats,
    cache,
    pitching_stats,
    standings,
    team_batting,
    team_pitching,
)

# Rely on pybaseball's built-in cache (version-safe API call).
cache.enable()

CURRENT_YEAR = date.today().year
YEARS_TO_PULL = [CURRENT_YEAR - 3, CURRENT_YEAR - 2, CURRENT_YEAR - 1]

# Set to True to include additional, potentially slower example pulls.
RUN_OPTIONAL_EXAMPLES = False

In [17]:
print( cache.config.cache_directory )

/Users/colettace/.pybaseball/cache


## Helper functions (PyBaseball primer utilities)


In [2]:
def normalize_years(years: List[int]) -> List[int]:
    """Return unique, sorted, valid MLB years."""
    valid_years = sorted({year for year in years if 1871 <= year <= CURRENT_YEAR})
    if not valid_years:
        raise ValueError('No valid years provided. Update YEARS_TO_PULL with MLB seasons.')
    return valid_years


def fetch_season_frames(year: int) -> Dict[str, pd.DataFrame]:
    """Download season-level hitter/pitcher leaderboards and attach season labels."""
    hitters_df = batting_stats(year, qual=0)
    pitchers_df = pitching_stats(year, qual=0)

    hitters_df = hitters_df.copy()
    pitchers_df = pitchers_df.copy()

    hitters_df['Season'] = year
    pitchers_df['Season'] = year

    return {'batting': hitters_df, 'pitching': pitchers_df}


def dataset_health(df: pd.DataFrame, dataset_name: str) -> Dict[str, object]:
    """Quick diagnostics to confirm data was downloaded and is queryable."""
    return {
        'dataset': dataset_name,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
        'null_cells': int(df.isna().sum().sum()),
        'sample_columns': ', '.join(df.columns[:8]),
    }


## Download three seasons of draft-relevant data with PyBaseball


In [3]:
years = normalize_years(YEARS_TO_PULL)

season_data: Dict[int, Dict[str, pd.DataFrame]] = {}
all_batting: List[pd.DataFrame] = []
all_pitching: List[pd.DataFrame] = []

for year in years:
    print(f'Downloading pybaseball leaderboards for {year}...')
    result = fetch_season_frames(year)
    season_data[year] = result

    all_batting.append(result['batting'])
    all_pitching.append(result['pitching'])

batting_all_years = pd.concat(all_batting, ignore_index=True)
pitching_all_years = pd.concat(all_pitching, ignore_index=True)

print('Download complete.')
print(f'Combined batting rows: {len(batting_all_years):,}')
print(f'Combined pitching rows: {len(pitching_all_years):,}')


Download complete.
Combined batting rows: 4,381
Combined pitching rows: 2,591


## Integrity checks: verify downloaded data can be queried


In [4]:
health_report = pd.DataFrame(
    [
        dataset_health(batting_all_years, 'batting_all_years'),
        dataset_health(pitching_all_years, 'pitching_all_years'),
    ]
)

health_report


,dataset,rows,columns,null_cells,sample_columns
0,batting_all_years,4381,320,718204,"IDfg, Season, Name, Team, Age, G, AB, PA"
1,pitching_all_years,2591,393,362180,"IDfg, Season, Name, Team, Age, W, L, WAR"


## Primer examples: common PyBaseball workflows


In [5]:
# 1) Top fantasy-relevant hitters by HR and SB from the most recent pulled season.
most_recent_year = years[-1]
recent_batting = season_data[most_recent_year]['batting']

recent_batting[['Name', 'Team', 'HR', 'SB', 'R', 'RBI', 'AVG']].sort_values(
    by=['HR', 'SB'], ascending=False
).head(15)

,Name,Team,HR,SB,R,RBI,AVG
13,Cal Raleigh,SEA,60,14,110,125,0.247
14,Kyle Schwarber,PHI,56,10,111,132,0.240
6,Shohei Ohtani,LAD,55,20,146,102,0.282
2,Aaron Judge,NYY,53,12,137,114,0.331
90,Eugenio Suarez,- - -,49,4,91,118,0.228
59,Junior Caminero,TBR,45,7,93,110,0.264
15,Juan Soto,NYM,43,38,120,105,0.263
30,Pete Alonso,NYM,38,1,87,126,0.272
140,Jo Adell,LAA,37,5,63,98,0.236
115,Taylor Ward,LAA,36,4,86,103,0.228


In [6]:
# 2) Top fantasy-relevant pitchers by strikeouts and saves from the most recent year.
recent_pitching = season_data[most_recent_year]['pitching']

recent_pitching[['Name', 'Team', 'W', 'SV', 'SO', 'ERA', 'WHIP']].sort_values(
    by=['SO', 'SV'], ascending=False
).head(15)

,Name,Team,W,SV,SO,ERA,WHIP
145,Garrett Crochet,BOS,18,0,255,2.59,1.03
99,Tarik Skubal,DET,13,0,241,2.21,0.89
253,Logan Webb,SFG,15,0,224,3.22,1.24
78,Paul Skenes,PIT,10,0,216,1.97,0.95
365,Jesus Luzardo,PHI,15,0,216,3.92,1.22
487,Dylan Cease,SDP,8,0,215,4.55,1.33
138,Cristopher Sanchez,PHI,13,0,212,2.50,1.06
121,Hunter Brown,HOU,12,0,206,2.43,1.03
159,Freddy Peralta,MIL,17,0,204,2.70,1.08
224,Carlos Rodon,NYY,18,0,203,3.09,1.05


In [7]:
# 3) Team-level snapshots often useful for draft context.
team_batting_recent = team_batting(most_recent_year)
team_pitching_recent = team_pitching(most_recent_year)

In [8]:
# Show small previews to verify these endpoints are usable.
display(team_batting_recent.head(10))
display(team_pitching_recent.head(10))

,teamIDfg,Season,Team,Age,G,AB,PA,H,1B,2B,...,maxEV,HardHit,HardHit%,Events,CStr%,CSW%,xBA,xSLG,xwOBA,L-WAR
0,9,2025,NYY,30,2401,5471,6235,1371,822,255,...,118.1,1879,0.461,4078,0.166,0.276,0.247,0.460,0.340,34.5
1,22,2025,LAD,30,2438,5481,6187,1384,862,257,...,120.0,1767,0.421,4198,0.167,0.277,0.251,0.447,0.335,28.9
2,14,2025,TOR,29,2481,5507,6180,1461,963,294,...,120.4,1847,0.411,4496,0.157,0.251,0.260,0.429,0.331,31.5
3,26,2025,PHI,30,2300,5517,6166,1426,922,268,...,117.2,1803,0.425,4241,0.150,0.264,0.252,0.432,0.328,26.4
4,15,2025,ARI,29,2332,5480,6210,1377,848,277,...,119.6,1754,0.411,4268,0.170,0.273,0.246,0.425,0.326,26.0
5,25,2025,NYM,30,2402,5457,6178,1359,854,262,...,115.9,1935,0.460,4203,0.166,0.271,0.255,0.450,0.339,29.5
6,17,2025,CHC,29,2330,5495,6162,1371,852,267,...,116.2,1712,0.400,4285,0.156,0.259,0.254,0.444,0.333,29.9
7,10,2025,ATH,28,2367,5547,6151,1403,872,296,...,115.5,1659,0.395,4199,0.152,0.271,0.239,0.403,0.310,20.6
8,3,2025,BOS,28,2371,5562,6206,1414,880,324,...,117.7,1882,0.448,4197,0.162,0.279,0.244,0.413,0.319,24.6
9,23,2025,MIL,28,2413,5510,6227,1423,974,265,...,114.1,1696,0.392,4324,0.180,0.272,0.246,0.388,0.314,26.7


,teamIDfg,Season,Team,Age,W,L,ERA,G,GS,CG,...,Pit+ FC,Stf+ FS,Loc+ FS,Pit+ FS,Stuff+,Location+,Pitching+,Stf+ FO,Loc+ FO,Pit+ FO
0,13,2025,TEX,31,81,81,3.49,685,162,2,...,107,94.0,104.0,104.0,101,102,103,NaN,NaN,NaN
1,23,2025,MIL,28,97,65,3.59,712,162,0,...,103,98.0,99.0,96.0,102,101,103,NaN,NaN,NaN
2,29,2025,SDP,29,90,72,3.64,739,162,2,...,99,106.0,100.0,99.0,103,100,103,NaN,NaN,NaN
3,5,2025,CLE,27,88,74,3.70,693,162,2,...,105,116.0,108.0,124.0,98,99,96,NaN,NaN,NaN
4,3,2025,BOS,29,89,73,3.72,698,162,2,...,109,102.0,104.0,109.0,102,100,101,NaN,NaN,NaN
5,7,2025,KCR,30,82,80,3.73,709,162,0,...,100,96.0,86.0,82.0,98,100,98,NaN,NaN,NaN
6,27,2025,PIT,28,71,91,3.76,672,162,1,...,105,97.0,107.0,107.0,101,99,100,NaN,NaN,NaN
7,26,2025,PHI,30,96,66,3.79,667,162,2,...,106,94.0,105.0,103.0,105,104,109,NaN,NaN,NaN
8,17,2025,CHC,30,92,70,3.81,685,162,0,...,99,88.0,101.0,90.0,98,102,99,NaN,NaN,NaN
9,30,2025,SFG,28,81,81,3.84,677,162,1,...,92,112.0,95.0,106.0,102,98,100,NaN,NaN,NaN


In [9]:
# 4) League standings are a useful contextual feature.
league_tables = standings(most_recent_year)
if league_tables:
    display(league_tables[0].head())

,Tm,W,L,W-L%,GB
1,Toronto Blue Jays,94,68,.580,--
2,New York Yankees,94,68,.580,--
3,Boston Red Sox,89,73,.549,5.0
4,Tampa Bay Rays,77,85,.475,17.0
5,Baltimore Orioles,75,87,.463,19.0
